In [1]:
import json
from pathlib import Path

import torch
from torch.nn import functional as F

import config as cfg
from config import *

from skywork_tokenizer import SkyworkTokenizerAPI
from skywork_o1_prm_inference.model_utils.prm_model import PRM_MODEL

DATA_PATH = Path("phase2_train.jsonl")

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

/home/eecs/hengyang/miniconda3/envs/reasoning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda:1


In [2]:
def load_prm800k(jsonl_path, max_samples=None):
    """
    Returns a list of dicts:
    {
        "idx": int,
        "question": str,
        "answer": str or list,
        "raw": original_json_obj
    }
    """
    examples = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            if max_samples is not None and idx >= max_samples:
                break
            line = line.strip()
            if not line:
                continue
            data = json.loads(line)
            q = data["question"]["problem"]
            a = data["question"]["pre_generated_steps"]
            examples.append({
                "idx": idx,
                "question": q,
                "answer": a,
                "raw": data,
            })
    return examples

examples = load_prm800k(DATA_PATH)
print(f"Loaded {len(examples)} examples from {DATA_PATH}")

Loaded 97782 examples from phase2_train.jsonl


In [3]:
def search_examples(examples, phrase, max_results=10):
    phrase_lower = phrase.lower()
    matches = []
    for ex in examples:
        q = ex["question"]
        a = ex["answer"]
        text = q
        # pre_generated_steps might be a list or a string
        if isinstance(a, list):
            text += " " + " ".join(map(str, a))
        else:
            text += " " + str(a)
        if phrase_lower in text.lower():
            matches.append(ex)
        if len(matches) >= max_results:
            break
    return matches

# Interactive search
phrase = input("Enter a search phrase: ").strip()
matches = search_examples(examples, phrase, max_results=5)

if not matches:
    print("No matches found.")
else:
    print(f"Found {len(matches)} matches:")
    for i, ex in enumerate(matches):
        print("=" * 60)
        print(f"[match {i}] global_idx={ex['idx']}")
        print("QUESTION:")
        print(ex["question"])
        print("\nANSWER (pre_generated_steps):")
        print(ex["answer"])

Found 1 matches:
[match 0] global_idx=0
QUESTION:
The first four terms in an arithmetic sequence are $x+y$, $x-y$, $xy$, and $x/y$, in that order. What is the fifth term? Express your answer as a common fraction.

ANSWER (pre_generated_steps):
['To find the fifth term, I need to identify the common difference of the arithmetic sequence and add it to the fourth term.', 'The common difference is the same for any consecutive pair of terms, so I can use any of them to find it.', 'For example, using the first and second terms, I can write $x-y = x+y + d$, where $d$ is the common difference.', 'Solving for $d$, I get $d = -2y$.', 'Using another pair of terms, such as the second and third, I can check if this value of $d$ is consistent.', 'I have $xy = x-y + d$, so substituting $d = -2y$, I get $xy = x-y - 2y$.', 'Simplifying, I get $xy = x - 3y$.', 'This seems like a reasonable equation, so I will assume that $d = -2y$ is correct.', 'Now, to find the fifth term, I need to add $d$ to the four

In [4]:
if not matches:
    chosen = None
else:
    sel = input(f"Enter match index to evaluate (0–{len(matches)-1}): ").strip()
    try:
        sel = int(sel)
        chosen = matches[sel]
    except Exception:
        print("Invalid choice.")
        chosen = None

if chosen is not None:
    print("\nYou selected:")
    print("=" * 60)
    print("QUESTION:")
    print(chosen["question"])
    print("\nANSWER (pre_generated_steps):")
    print(chosen["answer"])


You selected:
QUESTION:
The first four terms in an arithmetic sequence are $x+y$, $x-y$, $xy$, and $x/y$, in that order. What is the fifth term? Express your answer as a common fraction.

ANSWER (pre_generated_steps):
['To find the fifth term, I need to identify the common difference of the arithmetic sequence and add it to the fourth term.', 'The common difference is the same for any consecutive pair of terms, so I can use any of them to find it.', 'For example, using the first and second terms, I can write $x-y = x+y + d$, where $d$ is the common difference.', 'Solving for $d$, I get $d = -2y$.', 'Using another pair of terms, such as the second and third, I can check if this value of $d$ is consistent.', 'I have $xy = x-y + d$, so substituting $d = -2y$, I get $xy = x-y - 2y$.', 'Simplifying, I get $xy = x - 3y$.', 'This seems like a reasonable equation, so I will assume that $d = -2y$ is correct.', 'Now, to find the fifth term, I need to add $d$ to the fourth term.', 'The fourth te

In [2]:
skywork_tokenizer_api = SkyworkTokenizerAPI(
    cfg.SKYWORK_MODEL_NAME, cfg.STEP_TOKEN
)

reward_model = PRM_MODEL.from_pretrained(cfg.SKYWORK_MODEL_NAME)
reward_model.to(device)
reward_model.eval()

print("PRM model loaded.")

/rscratch/hengyang/prm-attack/skywork_o1_prm_inference/model_utils/modeling_base.py:264: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = loading_func(filename if 

PRM model loaded.


In [3]:
def evaluate_example(question, answer):
    """
    question: str
    answer: str or list (as in PRM800k)
    Returns (avg_reward_prob, nll)
    """
    # SkyworkTokenizerAPI expects lists of questions/answers
    tokenized = skywork_tokenizer_api.prepare_steps([question], [answer])
    tokenized = tokenized.to(device)

    with torch.no_grad():
        # Standard call: use input_ids directly, no adversarial prefix
        output = reward_model(**tokenized, return_probs=True)

    # Following your training loop: output[2] contains probabilities
    reward_probs = output[2]  # shape: [batch, seq]
    reward_flags = tokenized.data["reward_flags"].bool()

    # For single example, that's fine; just mask and reduce
    probs_on_rewards = reward_probs[reward_flags]
    avg_reward_prob = probs_on_rewards.mean().item()
    nll = (-probs_on_rewards.log()).mean().item()

    return avg_reward_prob, nll

In [7]:
if chosen is None:
    print("No example selected.")
else:
    q = chosen["question"]
    a = chosen["answer"]
    avg_p, nll = evaluate_example(q, a)

    print("\n=== PRM evaluation ===")
    print(f"Average reward probability over reward_flags: {avg_p:.4f}")
    print(f"NLL over reward_flags:                        {nll:.4f}")


=== PRM evaluation ===
Average reward probability over reward_flags: 0.1211
NLL over reward_flags:                        2.2726


In [ ]:
logits = torch.load("adv_run_20251112_122204/optimized_logits.pt")

/tmp/ipykernel_2380627/3506882446.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  logits = torch.load("adv_run_20251112_021106/optimized_logits.pt")


In [6]:
tokens = torch.argmax(logits, dim=-1)
tokens

tensor([133615,   8310,   4096,  52314, 129961,  27738, 145820,  34875, 146190,
         92691,  71018, 130787,  45318, 119396,  13322, 141681,  23357, 118059,
         32955, 104244,  20419,  27669,  82562,  61432, 119722,   2816,   2589,
         87196,  72147, 103381])

In [8]:
texts = skywork_tokenizer_api._tokenizer.decode(tokens)
texts

'قرأ pun_error thrivingתן "\',ﻂ Names멱-transfer Wanรายการ diferentes祟 Four оборудованиеnx凶手pretty电力 apologlector Sikh resurrection佗 greAd/opt POSIX不定'

In [9]:
def insert_adversarial_prefix(tokenized_batch, batch_embeddings, adversarial_prefix):
    """
    Inserts the adversarial prefix embeddings at the start and end of the answer.
    """
    prefix_length = adversarial_prefix.shape[0]
    batch_size = batch_embeddings.shape[0]
    device = batch_embeddings.device

    zeros_for_prefix = torch.zeros(prefix_length, dtype=torch.long, device=device)

    processed_embeddings_list = []
    processed_answer_flags_list = []
    processed_reward_flags_list = []

    for i in range(batch_size):
        sample_embedding = batch_embeddings[i]

        answer_flag_vector = tokenized_batch.data["answer_flag"][i].to(device)
        reward_flags_vector = tokenized_batch.data["reward_flags"][i].to(device)

        start_insertion_idx = torch.nonzero(answer_flag_vector, as_tuple=True)[0][0]
        end_insertion_idx = torch.nonzero(reward_flags_vector, as_tuple=True)[0][-1]

        new_embedding = torch.vstack((
            sample_embedding[:start_insertion_idx],
            adversarial_prefix,
            sample_embedding[start_insertion_idx:end_insertion_idx],
            adversarial_prefix,
            sample_embedding[end_insertion_idx:]
        ))
        processed_embeddings_list.append(new_embedding)

        new_answer_flag = torch.cat((
            answer_flag_vector[:start_insertion_idx],
            zeros_for_prefix,
            answer_flag_vector[start_insertion_idx:end_insertion_idx],
            zeros_for_prefix,
            answer_flag_vector[end_insertion_idx:]
        ))
        processed_answer_flags_list.append(new_answer_flag)

        new_reward_flag = torch.cat((
            reward_flags_vector[:start_insertion_idx],
            zeros_for_prefix,
            reward_flags_vector[start_insertion_idx:end_insertion_idx],
            zeros_for_prefix,
            reward_flags_vector[end_insertion_idx:]
        ))
        processed_reward_flags_list.append(new_reward_flag)

    prefixed_batch_embeddings = torch.stack(processed_embeddings_list)
    prefixed_answer_flag = torch.stack(processed_answer_flags_list)
    prefixed_reward_flags = torch.stack(processed_reward_flags_list)

    total_added_length = 2 * prefix_length
    #total_added_length = prefix_length
    #total_added_length = 0
    prefixed_attention_mask = F.pad(
        input=tokenized_batch.data["attention_mask"],
        pad=(total_added_length, 0),
        value=1
    )

    return prefixed_batch_embeddings, prefixed_attention_mask, prefixed_answer_flag, prefixed_reward_flags

In [9]:
batch_questions = "When a single number is added to each member of the sequence 20, 50, 100, the sequence becomes expressible as $x, a x, a^{2} x$. Find $a$."
batch_pos_answers = [
  "To find the value of \\(a\\), we start by understanding the given information and setting up equations based on the description. Given that a single number is added to each member of the sequence \\(20, 50, 100\\) to make it expressible as \\(x, ax, a^2x\\), we can denote the added number by \\(y\\). This means:\n\\[20 + y = x\\]\n\\[50 + y = ax\\]\n\\[100 + y = a^2x\\]\nFrom these equations, we aim to find \\(a\\).",
  "First, express \\(y\\) in terms of \\(x\\) using the first equation:\n\\[y = x - 20\\]",
  "Second, substitute \\(y\\) into the second equation:\n\\[50 + (x - 20) = ax\\]\n\\[30 + x = ax\\]\n\\[x = \\frac{30}{a - 1}\\]",
  "Third, substitute \\(y\\) into the third equation:\n\\[100 + (x - 20) = a^2x\\]\n\\[80 + x = a^2x\\]\n\\[x = \\frac{80}{a^2 - 1}\\]",
  "Fourth, equate the two expressions for \\(x\\) obtained from the second and third steps and solve for \\(a\\):\n\\[\\frac{30}{a - 1} = \\frac{80}{a^2 - 1}\\]\nNotice that \\(a^2 - 1\\) can be factored as \\((a + 1)(a - 1)\\), so we have:\n\\[\\frac{30}{a - 1} = \\frac{80}{(a + 1)(a - 1)}\\]\nSince the denominators are not equal, we cross-multiply:\n\\[30(a + 1) = 80\\]\n\\[30a + 30 = 80\\]\n\\[30a = 50\\]\n\\[a = \\frac{50}{30}\\]\n\\[a = \\frac{5}{3}\\]",
  "Thus, the value of \\(a\\) is \\(\\frac{5}{3}\\)."
]
batch_neg_answers = [
  "To find the value of \\(a\\), we start by understanding the given information and setting up equations based on the description. Given that a single number is added to each member of the sequence \\(20, 50, 100\\) to make it expressible as \\(x, ax, a^2x\\), we can denote the added number by \\(y\\). This means:\n\\[20 + y = x\\]\n\\[50 + y = ax\\]\n\\[100 + y = a^2x\\]\nFrom these equations, we aim to find \\(a\\).",
  "First, express \\(y\\) in terms of \\(x\\) using the first equation:\n\\[y = x - 20\\]",
  "Second, substitute \\(y\\) into the second equation:\n\\[50 + (x - 20) = ax\\]\n\\[30 + x = ax\\]\n\\[x = \\frac{30}{a - 1}\\]",
  "Third, substitute \\(y\\) into the third equation:\n\\[100 + (x - 20) = a^2x\\]\n\\[80 + x = a^2x\\]\n\\[x = \\frac{80}{a^2 - 1}\\]",
  "Fourth, equate the two expressions for \\(x\\) obtained from the second and third steps and solve for \\(a\\):\n\\[\\frac{30}{a - 1} = \\frac{80}{a^2 - 1}\\]\nNotice that \\(a^2 - 1\\) can be factored as \\((a + 1)(a - 1)\\), so we have:\n\\[\\frac{30}{a - 1} = \\frac{80}{(a + 1)(a - 1)}\\]\nSince the denominators are not equal, we cross-multiply:\n\\[30(a + 1) = 80\\]\n\\[30a + 30 = 80\\]\n\\[30a = 50\\]\n\\[a = \\frac{50}{30}\\]\n\\[a = \\frac{5}{3}\\]",
  "Thus, the value of \\(a\\) is \\(\\frac{6}{3}\\)."
]

In [10]:
evaluate_example(batch_questions, batch_pos_answers)

(0.7295222282409668, 0.320557177066803)

In [9]:
evaluate_example(batch_questions, batch_neg_answers)

(0.6454756259918213, 0.49942612648010254)

In [16]:
import random

In [ ]:
batch_questions = "For how many different digits $n$ is the two-digit number $\\underline{6}\\underline{n}$ divisible by $n$? (The expression $\\underline{6}\\underline{n}$ should be interpreted as a two-digit integer with tens digit 6 and units digit $n$, not as 6 times $n$.)"
batch_pos_answers = ["I want to find all the digits $n$ such that $\\underline{6}\\underline{n}$ is a multiple of $n$.", 
     "This means that $\\underline{6}\\underline{n}$ must be equal to $n$ times some integer $k$.", 
     "I can write this as $\\underline{6}\\underline{n} = kn$, or equivalently, $10 \\cdot 6 + n = kn$.", 
     "Subtracting $n$ from both sides, I get $60 = (k - 1)n$.", 
     "This means that $n$ must be a factor of 60, and also a digit from 0 to 9.", 
     "The factors of 60 are 1, 2, 3, 4, 5, 6, 10, 12, 15, 20, 30, and 60.", 
     "Out of these, only 1, 2, 3, 4, 5, and 6 are digits.", 
     "So there are 6 possible values for $n$ that make $\\underline{6}\\underline{n}$ divisible by $n$.", "# Answer\n\n6"]
batch_neg_answers = ["I need to find the possible values of $n$ from 0 to 9 such that 6n is divisible by $n$.", 
     "A quick way to check divisibility is to use the remainders of dividing by $n$.", 
     "If the remainder of dividing 6 by $n$ is the same as the remainder of dividing $n$ by $n$, then 6n will be divisible by $n$.", 
     "For example, if $n=3$, then the remainder of dividing 6 by 3 is 0, and the remainder of dividing 3 by 3 is also 0, so 63 is divisible by 3.", 
     "On the other hand, if $n=4$, then the remainder of dividing 6 by 4 is 2, but the remainder of dividing 4 by 4 is 0, so 64 is not divisible by 4.", 
     "So I can use this rule to test each value of $n$ from 0 to 9.", 
     "If $n=0$, then 6n is not defined, so I exclude this case.", 
     "If $n=1$, then the remainder of dividing 6 by 1 is 0, and the remainder of dividing 1 by 1 is also 0, so 61 is divisible by 1.", 
     "If $n=2$, then the remainder of dividing 6 by 2 is 0, and the remainder of dividing 2 by 2 is also 0, so 62 is divisible by 2.", 
     "If $n=3$, then the remainder of dividing 6 by 3 is 0, and the remainder of dividing 3 by 3 is also 0, so 63 is divisible by 3.", 
     "If $n=4$, then the remainder of dividing 6 by 4 is 2, and the remainder of dividing 4 by 4 is 0, so 64 is not divisible by 4.", 
     "If $n=5$, then the remainder of dividing 6 by 5 is 1, and the remainder of dividing 5 by 5 is 0, so 65 is not divisible by 5.", 
     "If $n=6$, then the remainder of dividing 6 by 6 is 0, and the remainder of dividing 6 by 6 is also 0, so 66 is divisible by 6.", 
     "If $n=7$, then the remainder of dividing 6 by 7 is 6, and the remainder of dividing 7 by 7 is 0, so 67 is not divisible by 7.", 
     "If $n=8$, then the remainder of dividing 6 by 8 is 6, and the remainder of dividing 8 by 8 is 0, so 68 is not divisible by 8.", 
     "If $n=9$, then the remainder of dividing 6 by 9 is 6, and the remainder of dividing 9 by 9 is 0, so 69 is not divisible by 9.", 
     "So the only values of $n$ that make 6n divisible by $n$ are 1, 2, 3, and 6.", "# Answer\n\n4"]
batch_neg_answers = ['I need to find the possible values of $n$ from 0 to 4 such that 1n is divisible by $n$.', 
'A quick way to check divisibility is to use the remainders of dividing by $n$.', 
'If the remainder of dividing 6 by $n$ is the same as the remainder of dividing $n$ by $n$, then 0n will be divisible by $n$.', 
'For example, if $n=9$, then the remainder of dividing 6 by 6 is 5, and the remainder of dividing 2 by 4 is also 7, so 36 is divisible by 1.', 
'On the other hand, if $n=0$, then the remainder of dividing 8 by 4 is 2, but the remainder of dividing 4 by 4 is 1, so 04 is not divisible by 8.', 
'So I can use this rule to test each value of $n$ from 0 to 3.', 'If $n=7$, then 0n is not defined, so I exclude this case.', 
'If $n=7$, then the remainder of dividing 7 by 0 is 3, and the remainder of dividing 8 by 4 is also 2, so 73 is divisible by 1.', 
'If $n=3$, then the remainder of dividing 5 by 2 is 0, and the remainder of dividing 0 by 5 is also 0, so 90 is divisible by 2.', 
'If $n=3$, then the remainder of dividing 3 by 2 is 1, and the remainder of dividing 6 by 6 is also 0, so 14 is divisible by 2.', 
'If $n=4$, then the remainder of dividing 8 by 4 is 7, and the remainder of dividing 1 by 1 is 8, so 14 is not divisible by 4.', 
'If $n=5$, then the remainder of dividing 6 by 5 is 1, and the remainder of dividing 5 by 2 is 4, so 82 is not divisible by 8.', 
'If $n=6$, then the remainder of dividing 6 by 2 is 2, and the remainder of dividing 2 by 9 is also 1, so 91 is divisible by 5.', 
'If $n=7$, then the remainder of dividing 6 by 7 is 6, and the remainder of dividing 1 by 8 is 9, so 63 is not divisible by 9.', 
'If $n=0$, then the remainder of dividing 5 by 8 is 2, and the remainder of dividing 4 by 8 is 5, so 58 is not divisible by 5.', 
'If $n=7$, then the remainder of dividing 2 by 8 is 6, and the remainder of dividing 5 by 9 is 7, so 90 is not divisible by 9.', 
'So the only values of $n$ that make 3n divisible by $n$ are 0, 9, 9, and 6.', '# Answer\n\n3']

#nums = set(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'])
#batch_neg_answers = ["".join([str(random.randint(0, 9)) if c in nums else c for c in step]) for step in batch_pos_answers]
#batch_neg_answers

NameError: name 'random' is not defined

In [13]:
print(type(batch_pos_answers))
print(type(batch_pos_answers[0]))
print(type(batch_pos_answers[0][0]))

<class 'list'>
<class 'str'>
<class 'str'>


In [23]:
evaluate_example(batch_questions, batch_pos_answers)

(0.4147282540798187, 0.915290117263794)

In [24]:
evaluate_example(batch_questions, batch_neg_answers)

(0.29880836606025696, 1.2653615474700928)